# 06 — LIMU-BERT Held-Out Testing & Feature Evaluation

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Section 16:** Testing notebook must load the saved checkpoint and evaluate held-out data.
> **Section 9:** Trajectory/session-level test split (Driver A session `S1` kept strictly unseen during training).

### Objectives:
1. Load pretrained checkpoint from `checkpoints/limu_bert/limu_bert_best.pt`.
2. Evaluate masked sensor reconstruction error on completely held-out test session `S1`.
3. Inspect channel-by-channel reconstruction fidelity (3-axis Accel + 3-axis Gyro).
4. Verify latent feature representation quality for downstream inertial odometry.

## 1. Imports & Checkpoint Loading

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.limu_bert import LIMUBERT
from src.datasets.limu_bert_dataset import LIMUBERTDataset
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = PROJECT_ROOT / 'checkpoints' / 'limu_bert' / 'limu_bert_best.pt'
plots_dir = PROJECT_ROOT / 'plots' / 'limu_bert'
plots_dir.mkdir(parents=True, exist_ok=True)

print(f'Loading checkpoint: {ckpt_path}')
if not ckpt_path.exists():
    raise FileNotFoundError(f'Checkpoint not found at {ckpt_path}. Run 05_limu_bert_training.ipynb on Lightning first.')

ckpt = torch.load(ckpt_path, map_location=device)
cfg = ckpt.get('config', {})

model = LIMUBERT(
    input_dim=6,
    hidden_dim=cfg.get('hidden_dim', 128),
    num_heads=cfg.get('num_heads', 4),
    num_layers=cfg.get('num_layers', 4)
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'LIMU-BERT successfully loaded from Epoch {ckpt.get("epoch")} (Best Val Loss: {ckpt.get("best_val_loss"):.5f})')

## 2. Load Held-Out Test Dataset (Session S1 — Driver A)

In [ ]:
test_ds = LIMUBERTDataset(
    split='test',
    window_size=120,
    stride=60,
    mask_ratio=0.15
)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
print(f'Test dataset samples: {len(test_ds)} windows')

## 3. Evaluate Reconstruction Loss on Held-Out Data

In [ ]:
total_test_loss = 0.0
n_batches = 0
all_preds = []
all_targets = []
all_masks = []

with torch.no_grad():
    for batch in test_loader:
        x_in = batch['input'].to(device)
        target = batch['target'].to(device)
        mask = batch['mask'].to(device)
        
        recon, _ = model(x_in)
        loss = model.compute_pretraining_loss(recon, target, mask)
        total_test_loss += loss.item()
        n_batches += 1
        
        all_preds.append(recon.cpu().numpy())
        all_targets.append(target.cpu().numpy())
        all_masks.append(mask.cpu().numpy())

avg_test_loss = total_test_loss / max(1, n_batches)
print('=' * 60)
print(f'HELD-OUT TEST SET (DRIVER A - S1) RECONSTRUCTION MSE: {avg_test_loss:.5f}')
print('=' * 60)

## 4. Visualize Masked vs Reconstructed Sensor Signals

In [ ]:
sample_idx = 0
pred_w = all_preds[0][sample_idx]      # (120, 6)
target_w = all_targets[0][sample_idx]  # (120, 6)
mask_w = all_masks[0][sample_idx]      # (120,)

t_axis = np.linspace(0, 12.0, 120)
channels = ['Accel X (m/s²)', 'Accel Y (m/s²)', 'Accel Z (m/s²)', 'Gyro X (rad/s)', 'Gyro Y (rad/s)', 'Gyro Z (rad/s)']

fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)
for c, ax in enumerate(axes.flat):
    ax.plot(t_axis, target_w[:, c], 'k-', label='True Signal', alpha=0.7)
    ax.plot(t_axis, pred_w[:, c], 'b--', label='LIMU-BERT Reconstruction', linewidth=1.5)
    
    # Highlight masked areas
    masked_intervals = np.where(mask_w)[0]
    if len(masked_intervals) > 0:
        for idx in masked_intervals:
            ax.axvspan(t_axis[idx]-0.05, t_axis[idx]+0.05, color='salmon', alpha=0.3)
            
    ax.set_ylabel(channels[c], fontweight='bold')
    if c == 0:
        ax.legend(loc='upper right')
    ax.grid(True, alpha=0.4)

axes[-1, 0].set_xlabel('Time [s]', fontweight='bold')
axes[-1, 1].set_xlabel('Time [s]', fontweight='bold')
fig.suptitle('LIMU-BERT Held-Out Reconstruction on Session S1 (Shaded = Masked Timesteps)', fontsize=13, fontweight='bold')
plt.tight_layout()

plot_out = plots_dir / 'limu_bert_test_reconstruction_S1.png'
plt.savefig(plot_out, dpi=200)
plt.close()
print(f'Test reconstruction plot saved: {plot_out}')

# Save held-out results
res = {
    'session': 'S1',
    'split': 'held_out_test',
    'reconstruction_mse': round(float(avg_test_loss), 6),
    'status': 'VERIFIED'
}
with open(PROJECT_ROOT / 'results' / 'limu_bert_test_results.json', 'w') as f:
    json.dump(res, f, indent=2)
print('Test results saved to results/limu_bert_test_results.json.')